# 01 — Setup Credentials (GCP)

Validate gcloud auth, confirm SDK versions, and initialize `config.json` for downstream notebooks.

AWS-side equivalent: `notebooks/aws/01-setup-credentials.ipynb` (AWS SSO + boto3).


## Step 1: Confirm gcloud is authenticated


In [ ]:
import subprocess
out = subprocess.run(['gcloud', 'auth', 'list', '--format=value(account,status)'], capture_output=True, text=True)
print(out.stdout or '(no accounts)')
if out.returncode != 0:
    print('STDERR:', out.stderr)


## Step 2: Confirm ADC (Application Default Credentials) are available

The Python SDKs (Vertex AI, Discovery Engine, etc.) look for ADC. If this fails, run:

```
gcloud auth application-default login
```


In [ ]:
from google.auth import default
from google.auth.exceptions import DefaultCredentialsError
try:
    creds, project = default()
    print(f'✓ ADC found; default project = {project}')
except DefaultCredentialsError as e:
    print(f'✗ ADC missing: {e}')
    print('  → run `gcloud auth application-default login` in your host shell')


## Step 3: Active project + region


In [ ]:
import os
PROJECT_ID = os.environ.get('GCP_PROJECT_ID') or subprocess.run(
    ['gcloud', 'config', 'get-value', 'project'], capture_output=True, text=True
).stdout.strip()
REGION = os.environ.get('GCP_REGION', 'us-central1')
print(f'project: {PROJECT_ID}')
print(f'region : {REGION}')
assert PROJECT_ID, 'GCP_PROJECT_ID is unset and gcloud has no active project — set one before continuing.'


## Step 4: Check required APIs are enabled


In [ ]:
REQUIRED = [
    'aiplatform.googleapis.com',
    'discoveryengine.googleapis.com',
    'alloydb.googleapis.com',
    'cloudtrace.googleapis.com',
    'modelarmor.googleapis.com',
]
enabled = subprocess.run(
    ['gcloud', 'services', 'list', '--enabled', '--format=value(name)'],
    capture_output=True, text=True,
).stdout.split()
missing = [s for s in REQUIRED if s not in enabled]
if missing:
    print('✗ Missing APIs (enable with `gcloud services enable ...`):')
    for m in missing: print(f'   {m}')
else:
    print('✓ All required APIs enabled.')


## Step 5: Validate SDK versions


In [ ]:
import importlib.metadata as md
for pkg in ['google-cloud-aiplatform', 'vertexai', 'google-genai', 'google-cloud-discoveryengine']:
    try:
        print(f'  {pkg:35s} {md.version(pkg)}')
    except md.PackageNotFoundError:
        print(f'  {pkg:35s} (not installed)')


## Step 6: Initialize / update config.json

Writes the GCP context under `cfg['gcp']` so downstream notebooks can read it.


In [ ]:
import json
from pathlib import Path

config_path = Path('config.json')
cfg = json.loads(config_path.read_text()) if config_path.exists() else {}
cfg.setdefault('gcp', {})
cfg['gcp']['project_id'] = PROJECT_ID
cfg['gcp']['region'] = REGION
cfg['gcp'].setdefault('tracked_resources', [])
config_path.write_text(json.dumps(cfg, indent=2))
print(f'✓ config.json updated with gcp.project_id={PROJECT_ID}, gcp.region={REGION}')


## Step 7: Smoke-test Vertex AI

Calls `text-embedding-005` with a tiny input. Confirms ADC + API enablement + region in one shot.


In [ ]:
from vertexai import init as vai_init
from vertexai.language_models import TextEmbeddingModel

vai_init(project=PROJECT_ID, location=REGION)
model = TextEmbeddingModel.from_pretrained('text-embedding-005')
emb = model.get_embeddings(['OrgIQ is an organizational intelligence platform.'])
print(f'✓ embedding ok — dim={len(emb[0].values)}')


## Done

Move on to `02-alloydb-graph.ipynb`.
